# Exam Score Prediction – Bayesian CV RMSE Notebook

## Configuration and Libraries Loading

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import optuna

RANDOM_STATE = 42

import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

## Load and QC Data

In [2]:
# data
path        = '/kaggle/input/playground-series-s6e1/'
df_train       = pd.read_csv(path + 'train.csv',             index_col = 'id')
df_test        = pd.read_csv(path + 'test.csv',              index_col = 'id')

df_train.describe().T

,count,mean,std,min,25%,50%,75%,max
age,630000.0,20.545821,2.260238,17.000,19.00,21.0,23.00,24.00
study_hours,630000.0,4.002337,2.359880,0.080,1.97,4.0,6.05,7.91
class_attendance,630000.0,71.987261,17.430098,40.600,57.00,72.6,87.20,99.40
sleep_hours,630000.0,7.072758,1.744811,4.100,5.60,7.1,8.60,9.90
exam_score,630000.0,62.506672,18.916884,19.599,48.80,62.6,76.30,100.00


## Define Target and Features

In [16]:
# Define Target & Features
TARGET = "exam_score"

X = df_train.drop(columns=[TARGET])
y = df_train[TARGET]

## Identify Numerical & Categorical Columns

In [17]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Numerical columns: ['age', 'study_hours', 'class_attendance', 'sleep_hours']
Categorical columns: ['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']


## Pre-processing pipeline

In [18]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)


## Split and Train Data 

In [19]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

## Model Defintion

In [20]:
models = {
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, random_state=RANDOM_STATE
    ),
    "XGBoost": XGBRegressor(
        n_estimators=300, learning_rate=0.025, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=300, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE
    )
}

## RMSE Evaluation

In [22]:
from sklearn.metrics import root_mean_squared_error

results = []

for name, model in models.items():
    # Create the pipeline
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    # Train and predict
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_valid)
    
    # Calculate RMSE using the updated function
    rmse = root_mean_squared_error(y_valid, preds)
    
    # Store results
    results.append({"Model": name, "RMSE": rmse})
    print(f"{name:<18} RMSE: {rmse:.4f}")

# Convert to DataFrame and reset index for a cleaner look
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
results_df

Ridge              RMSE: 8.8865
RandomForest       RMSE: 9.0603
GradientBoosting   RMSE: 8.8174
XGBoost            RMSE: 8.7666
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019165 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 625
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 30
[LightGBM] [Info] Start training from score 62.482335
LightGBM           RMSE: 8.7550


,Model,RMSE
0,LightGBM,8.754999
1,XGBoost,8.766644
2,GradientBoosting,8.817373
3,Ridge,8.886490
4,RandomForest,9.060313


## Cross Validation

In [24]:
def cv_rmse(pipeline, X, y, n_splits=5):
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(
        pipeline,
        X,
        y,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1
    )
    return -scores.mean()


## Bayesian Optimization – XGBoost

In [26]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 900),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 2.0),
        "objective": "reg:squarederror",
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
    
    model = XGBRegressor(**params)
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    return cv_rmse(pipe, X_train, y_train)

study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(objective_xgb, n_trials=12)

study_xgb.best_value, study_xgb.best_params


[I 2026-01-01 09:21:13,834] A new study created in memory with name: no-name-7db7ddad-43f9-448e-80f8-b70641ecb527
[I 2026-01-01 09:22:44,548] Trial 0 finished with value: 8.76892734432636 and parameters: {'n_estimators': 478, 'max_depth': 8, 'learning_rate': 0.05564425183340915, 'subsample': 0.952775635669968, 'colsample_bytree': 0.6005352576908413, 'reg_alpha': 0.9434055588660406, 'reg_lambda': 1.9512114290373375}. Best is trial 0 with value: 8.76892734432636.
[I 2026-01-01 09:24:19,443] Trial 1 finished with value: 8.773638703299625 and parameters: {'n_estimators': 563, 'max_depth': 7, 'learning_rate': 0.030133356461339196, 'subsample': 0.8809366484770533, 'colsample_bytree': 0.9406028317063819, 'reg_alpha': 0.21964051291351605, 'reg_lambda': 0.9054490937743176}. Best is trial 0 with value: 8.76892734432636.
[I 2026-01-01 09:25:49,930] Trial 2 finished with value: 8.79189986768721 and parameters: {'n_estimators': 438, 'max_depth': 7, 'learning_rate': 0.017899086303690196, 'subsample'

(8.768240929735892,
 {'n_estimators': 749,
  'max_depth': 8,
  'learning_rate': 0.037199929724759315,
  'subsample': 0.8921711958626668,
  'colsample_bytree': 0.9374767659386988,
  'reg_alpha': 0.7404813405637447,
  'reg_lambda': 0.9667995970473604})

## Bayesian Optimization – LightGBM

In [29]:
def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "num_leaves": trial.suggest_int("num_leaves", 20, 80),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 2.0),
        "random_state": RANDOM_STATE,
        "verbosity": -1
    }
    
    model = LGBMRegressor(**params)
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    return cv_rmse(pipe, X_train, y_train)

study_lgbm = optuna.create_study(direction="minimize")
study_lgbm.optimize(objective_lgbm, n_trials=12)

study_lgbm.best_value, study_lgbm.best_params


[I 2026-01-01 09:50:29,288] A new study created in memory with name: no-name-5082e239-f44c-4228-8d3c-467f5eeedbe9
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LG

(8.765708569520143,
 {'n_estimators': 1143,
  'num_leaves': 51,
  'learning_rate': 0.05347309192427835,
  'subsample': 0.8350871068626026,
  'colsample_bytree': 0.7677942058623412,
  'min_child_samples': 13,
  'reg_alpha': 0.6873822221359284,
  'reg_lambda': 1.7525433215806727})

## Select Best Model

In [30]:
best_model_name = "XGBoost" if study_xgb.best_value < study_lgbm.best_value else "LightGBM"
best_params = study_xgb.best_params if best_model_name == "XGBoost" else study_lgbm.best_params

best_model = (
    XGBRegressor(**best_params, objective="reg:squarederror", random_state=RANDOM_STATE)
    if best_model_name == "XGBoost"
    else LGBMRegressor(**best_params, random_state=RANDOM_STATE)
)

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model)
])

final_pipeline.fit(X, y)

print(f"✅ Final Model: {best_model_name}")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020004 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 625
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 30
[LightGBM] [Info] Start training from score 62.506672
✅ Final Model: LightGBM


## Final Prediction on whole dataset

In [33]:
df_train["predicted_exam_score"] = final_pipeline.predict(X)
df_train[[TARGET, "predicted_exam_score"]].head()


,exam_score,predicted_exam_score
id,,
0,78.3,84.940117
1,46.7,63.334642
2,99.0,75.100764
3,63.9,47.400656
4,100.0,93.876372


## Generate Prediction on test dataset

In [56]:

test_predictions = final_pipeline.predict(df_test)

df_test["exam_score"] = test_predictions

df_test.head()


,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,predicted_exam_score,exam_score
id,,,,,,,,,,,,,
630000,24,other,ba,6.85,65.2,yes,5.2,poor,group study,high,easy,71.242511,71.242511
630001,18,male,diploma,6.61,45.0,no,9.3,poor,coaching,low,easy,70.906151,70.906151
630002,24,female,b.tech,6.60,98.5,yes,6.2,good,group study,medium,moderate,88.190358,88.190358
630003,24,male,diploma,3.03,66.3,yes,5.7,average,mixed,medium,moderate,57.010633,57.010633
630004,20,female,b.tech,2.03,42.4,yes,9.2,average,coaching,low,moderate,47.563316,47.563316


In [53]:
print(df_test.columns.tolist())

['age', 'gender', 'course', 'study_hours', 'class_attendance', 'internet_access', 'sleep_hours', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty', 'predicted_exam_score']


## Save the output

In [57]:
submission  = pd.read_csv(path + 'sample_submission.csv', index_col = 'id')

submission['exam_score'] = test_predictions
submission.to_csv('submission.csv')
submission.head(10)

submission.head()

,exam_score
id,
630000,71.242511
630001,70.906151
630002,88.190358
630003,57.010633
630004,47.563316
